In [ ]:
#setup
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
pd.options.display.precision = 4
rng = np.random.default_rng(42)
print(sys.executable)

In [ ]:
#a skewed population, and the means of its samples
def sample_means(draw, n, repeats=20_000):
    """Draw `repeats` samples of size n and return the mean of each one."""
    return draw(size=(repeats, n)).mean(axis=1)

exp_draw = lambda size: rng.exponential(5, size)          # population: exponential, mean 5, SD 5 (right-skewed)
means = {n: sample_means(exp_draw, n) for n in (1, 2, 5, 30, 100)}

rows = [{"n": n, "mean of means": m.mean(), "SD of means": m.std(ddof=1),
         "sigma/sqrt(n)": 5 / math.sqrt(n), "skew of means": stats.skew(m), "2/sqrt(n)": 2 / math.sqrt(n)}
        for n, m in means.items()]
print(pd.DataFrame(rows).set_index("n").round(3))

In [ ]:
#watching the shape change
fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
for ax, (n, m) in zip(axes, means.items()):
    ax.hist(m, bins=50, density=True, color="lightsteelblue", edgecolor="white")
    xs = np.linspace(m.min(), m.max(), 300)
    ax.plot(xs, stats.norm.pdf(xs, 5, 5 / math.sqrt(n)), color="tomato", lw=2)
    ax.set_title(f"means of n = {n}")
plt.tight_layout()
plt.show()

In [ ]:
#five very different populations
populations = {
    "uniform(0, 1)": lambda size: rng.uniform(0, 1, size),
    "exponential": lambda size: rng.exponential(1, size),
    "die roll": lambda size: rng.integers(1, 7, size),
    "bimodal": lambda size: np.where(rng.random(size) < 0.5, rng.normal(0, 1, size), rng.normal(10, 1, size)),
    "lognormal(0, 1)": lambda size: rng.lognormal(0, 1, size),
}

fig, axes = plt.subplots(2, 5, figsize=(16, 5))
for col, (name, draw) in enumerate(populations.items()):
    axes[0, col].hist(draw(20_000), bins=50, color="lightsteelblue", edgecolor="white")
    axes[0, col].set_title(name)
    m = sample_means(draw, 30, 10_000)
    axes[1, col].hist(m, bins=50, density=True, color="lightgreen", edgecolor="white")
    xs = np.linspace(m.min(), m.max(), 300)
    axes[1, col].plot(xs, stats.norm.pdf(xs, m.mean(), m.std()), color="tomato", lw=2)
    axes[1, col].set_title("means of n = 30")
plt.tight_layout()
plt.show()

In [ ]:
#measuring “how normal” the means are
def ks_distance(means):
    """Largest gap between the standardised means' CDF and the normal CDF. 0 = perfectly normal."""
    z = (means - means.mean()) / means.std()
    return stats.kstest(z, "norm").statistic

table = {name: {n: ks_distance(sample_means(draw, n, 10_000)) for n in (1, 2, 5, 30, 100)}
         for name, draw in populations.items()}
print(pd.DataFrame(table).T.round(3))

In [ ]:
#a Q-Q plot of the means
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, n in zip(axes, (2, 10, 100)):
    stats.probplot(sample_means(exp_draw, n, 5000), dist="norm", plot=ax)
    ax.set_title(f"Q-Q plot of exponential means, n = {n}")
plt.tight_layout()
plt.show()

In [ ]:
#the CLT for sums
sums = rng.exponential(5, size=(20_000, 50)).sum(axis=1)
print(f"sum of 50 waits: mean {sums.mean():.1f} (expected {50 * 5}), SD {sums.std():.2f} (expected {5 * math.sqrt(50):.2f})")

In [ ]:
#SE from a single sample
sample = rng.exponential(5, 40)
s = sample.std(ddof=1)
print("SD of the data           :", round(s, 3))
print("SE of the mean (s/sqrt n):", round(s / math.sqrt(len(sample)), 3), "| stats.sem:", round(stats.sem(sample), 3))

print("\nSE as n grows (sigma = 5):")
for n in (25, 100, 400, 1600):
    print(f"  n = {n:>4}: SE = {5 / math.sqrt(n):.3f}")

In [ ]:
#hree worked examples
# 1. Waiting times are exponential with mean 5 and SD 5. What is P(the average of 50 waits > 6)?
mu, sigma, n = 5, 5, 50
se = sigma / math.sqrt(n)
approx = stats.norm.sf(6, loc=mu, scale=se)
exact = stats.gamma.sf(6 * n, a=n, scale=5)             # a sum of n exponentials follows a gamma distribution
simulated = (rng.exponential(5, size=(200_000, n)).mean(axis=1) > 6).mean()
print(f"1. CLT {approx:.4f} | exact {exact:.4f} | simulated {simulated:.4f}   (z = {(6 - mu) / se:.3f})")

# 2. People weigh 70 kg on average (SD 12). A lift carries 25 people. What is P(the total exceeds 1850 kg)?
total_mu, total_sd = 25 * 70, 12 * math.sqrt(25)
print(f"2. total ~ N({total_mu}, {total_sd}):  P(total > 1850) = {stats.norm.sf(1850, total_mu, total_sd):.4f}")

# 3. A fair coin is flipped 100 times. What is P(60 or more heads)?
exact = stats.binom.sf(59, 100, 0.5)
no_correction = stats.norm.sf(60, 50, 5)
with_correction = stats.norm.sf(59.5, 50, 5)             # continuity correction: 60 covers 59.5 to 60.5
print(f"3. exact {exact:.4f} | normal {no_correction:.4f} | normal with continuity correction {with_correction:.4f}")

In [ ]:
#z vs t critical values
print(f"z* for 95%: {stats.norm.ppf(0.975):.3f}\n")
for df in (2, 5, 10, 30, 100, 1000):
    print(f"t* for 95%, df = {df:>4}: {stats.t.ppf(0.975, df):.3f}")

In [ ]:
#one confidence interval
heights = rng.normal(170, 10, 25)
xbar, s, n = heights.mean(), heights.std(ddof=1), len(heights)
se = s / math.sqrt(n)
t_star = stats.t.ppf(0.975, df=n - 1)

print(f"mean {xbar:.2f}, s {s:.2f}, SE {se:.2f}, t* {t_star:.3f}")
print(f"margin of error: {t_star * se:.2f}")
print(f"95% CI by hand      : ({xbar - t_star * se:.2f}, {xbar + t_star * se:.2f})")
print("95% CI by scipy     :", np.round(stats.t.interval(0.95, n - 1, loc=xbar, scale=se), 2))
print("true mean was 170: is it inside?", stats.t.interval(0.95, n - 1, loc=xbar, scale=se)[0] <= 170 <= stats.t.interval(0.95, n - 1, loc=xbar, scale=se)[1])

In [ ]:
#what “95% confident” really means
reps, n = 40, 25
x = rng.normal(170, 10, size=(reps, n))
xbar = x.mean(axis=1)
half = stats.t.ppf(0.975, n - 1) * x.std(axis=1, ddof=1) / math.sqrt(n)
missed = (170 < xbar - half) | (170 > xbar + half)

fig, ax = plt.subplots(figsize=(7, 6))
for i in range(reps):
    ax.plot([xbar[i] - half[i], xbar[i] + half[i]], [i, i], color="tomato" if missed[i] else "steelblue", lw=2)
    ax.plot(xbar[i], i, "o", color="black", ms=3)
ax.axvline(170, color="black", ls="--")
ax.set_xlabel("height (cm)")
ax.set_ylabel("sample number")
ax.set_title("40 different samples, 40 different intervals")
plt.show()
print(missed.sum(), "of", reps, "intervals missed the true mean (expect about 2)")

In [ ]:
#he coverage of z and t intervals
def coverage(n, method, reps=20_000, mu=170, sigma=10, level=0.95):
    x = rng.normal(mu, sigma, size=(reps, n))
    xbar = x.mean(axis=1)
    se = x.std(axis=1, ddof=1) / math.sqrt(n)
    crit = stats.t.ppf((1 + level) / 2, n - 1) if method == "t" else stats.norm.ppf((1 + level) / 2)
    return np.mean((xbar - crit * se <= mu) & (mu <= xbar + crit * se))

for n in (5, 10, 30, 100):
    print(f"n = {n:>3}: z-interval {coverage(n, 'z'):.3f}   t-interval {coverage(n, 't'):.3f}   (target 0.95)")

In [ ]:
#what controls the width?
sigma = 15
print("margin of error (95%) for sigma = 15:")
for n in (25, 100, 400):
    print(f"  n = {n:>3}: +/- {1.96 * sigma / math.sqrt(n):.2f}")
print("\nconfidence level (n = 100):")
for level in (0.90, 0.95, 0.99):
    print(f"  {level:.0%}: +/- {stats.norm.ppf((1 + level) / 2) * sigma / math.sqrt(100):.2f}")

In [ ]:
#how much data do I need?
#Solve the margin of error formula for n: n = (z* σ / E)².
sigma, E = 15, 2                                        # estimate the mean to within +/- 2 units
for level in (0.90, 0.95, 0.99):
    z = stats.norm.ppf((1 + level) / 2)
    print(f"{level:.0%} confidence: z* = {z:.3f}, n = {math.ceil((z * sigma / E) ** 2)}")

In [ ]:
#confidence interval for a proportion, and why polls ask about 1,000 people
n, successes = 1000, 520
p_hat = successes / n
se = math.sqrt(p_hat * (1 - p_hat) / n)
low, high = p_hat - 1.96 * se, p_hat + 1.96 * se
print(f"p-hat {p_hat:.3f}, SE {se:.4f}, 95% CI ({low:.3f}, {high:.3f})")
print("is 50% a plausible value?", low <= 0.5 <= high, "-> we cannot claim a majority")

E = 0.03                                                # a margin of error of +/- 3 points; p = 0.5 is the worst case
print("sample size for +/-3%:", math.ceil((1.96 / E) ** 2 * 0.25))
print("a population of 1.4 billion needs about the same n, because SE depends on n, not on the population size")

In [ ]:
#he bootstrap, a CI without formulas
data = rng.lognormal(3, 0.8, 60)                        # skewed data; a formula for the median's CI is awkward
boot = np.array([np.median(rng.choice(data, size=len(data), replace=True)) for _ in range(10_000)])
print("sample median     :", np.median(data).round(2))
print("bootstrap 95% CI  :", np.percentile(boot, [2.5, 97.5]).round(2))
res = stats.bootstrap((data,), np.median, confidence_level=0.95, random_state=1)
print("scipy's bootstrap :", np.round([res.confidence_interval.low, res.confidence_interval.high], 2))

In [ ]:
#a population with no finite variance (Cauchy)
print(f"{'n':>8} {'IQR of the means (Cauchy)':>28} {'IQR of the means (normal)':>28}")
for n in (10, 1000, 100_000):
    cauchy_means = np.array([rng.standard_cauchy(n).mean() for _ in range(200)])
    normal_means = np.array([rng.standard_normal(n).mean() for _ in range(200)])
    print(f"{n:>8} {stats.iqr(cauchy_means):>28.3f} {stats.iqr(normal_means):>28.4f}")

In [ ]:
#dependent data
def correlated_means(n, rho=0.5, reps=20_000):
    shared = rng.normal(0, 1, size=(reps, 1))               # a component every observation shares
    noise = rng.normal(0, 1, size=(reps, n))
    return (math.sqrt(rho) * shared + math.sqrt(1 - rho) * noise).mean(axis=1)

print(f"{'n':>6} {'SD of mean if independent':>28} {'SD of mean if correlated':>26}")
for n in (10, 100, 1000):
    print(f"{n:>6} {1 / math.sqrt(n):>28.3f} {correlated_means(n).std():>26.3f}")

In [ ]:
#a small sample from a skewed population
def coverage_exponential(n, reps=20_000):
    x = rng.exponential(5, size=(reps, n))
    xbar = x.mean(axis=1)
    se = x.std(axis=1, ddof=1) / math.sqrt(n)
    crit = stats.t.ppf(0.975, n - 1)
    return np.mean((xbar - crit * se <= 5) & (5 <= xbar + crit * se))

for n in (10, 30, 100, 400):
    print(f"n = {n:>3}: 95% t-interval actually covers the true mean {coverage_exponential(n):.3f} of the time")

In [ ]:
#Mini project, “How long do deliveries really take?”
#I only have a sample of 200 deliveries from a population of 50,000, build confidence intervals, and then compare them with the truth

#create and save the data
from pathlib import Path

SCRATCH = Path.cwd() / "scratch"                                  # relative to the notebook's folder
SCRATCH.mkdir(exist_ok=True)

population = rng.lognormal(mean=3.2, sigma=0.5, size=50_000).round(1)       # minutes: right-skewed
true_mean, true_median = population.mean(), np.median(population)
survey = pd.DataFrame({"order_id": range(1, 201), "minutes": rng.choice(population, 200, replace=False)})
survey.to_csv(SCRATCH / "delivery_sample.csv", index=False)
print(f"(hidden truth: mean {true_mean:.2f}, median {true_median:.2f})")

In [ ]:
#look before we calculate
df = pd.read_csv(SCRATCH / "delivery_sample.csv")
print(df["minutes"].describe().round(2))
print("skew:", round(stats.skew(df["minutes"]), 2))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].hist(df["minutes"], bins=25, color="lightsteelblue", edgecolor="white")
axes[0].axvline(df["minutes"].mean(), color="tomato", lw=2)
axes[0].axvline(df["minutes"].median(), color="seagreen", lw=2, ls="--")
axes[0].set_title("minutes (right-skewed)")
stats.probplot(df["minutes"], dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot of minutes")
stats.probplot(np.log(df["minutes"]), dist="norm", plot=axes[2])
axes[2].set_title("Q-Q plot of log(minutes)")
plt.tight_layout()
plt.show()

In [ ]:
#Confidence intervals at three sample sizes
def mean_ci(x, level=0.95):
    x = np.asarray(x)
    return stats.t.interval(level, len(x) - 1, loc=x.mean(), scale=stats.sem(x))

for n in (30, 100, 200):
    sub = df["minutes"].sample(n, random_state=1)
    low, high = mean_ci(sub)
    print(f"n = {n:>3}: mean {sub.mean():5.2f}   95% CI ({low:5.2f}, {high:5.2f})   width {high - low:5.2f}   "
          f"contains the true mean ({true_mean:.2f})? {low <= true_mean <= high}")

log_low, log_high = np.exp(mean_ci(np.log(df["minutes"])))
print(f"\ngeometric mean {stats.gmean(df['minutes']):.2f} minutes, 95% CI ({log_low:.2f}, {log_high:.2f}) "
      f"(a good estimate of the typical delivery; the true median is {true_median:.2f})")

boot = np.array([np.median(rng.choice(df["minutes"], 200, replace=True)) for _ in range(5000)])
print("median 95% bootstrap CI:", np.percentile(boot, [2.5, 97.5]).round(2))

In [ ]:
#conclusion
## Findings

# We timed 200 of the company's deliveries.
#The average delivery took about __ minutes. We are 95% confident, from this sample, that the
#true average across all deliveries lies between __ and __ minutes.
#A typical delivery (the median) takes about __ minutes. The average is higher because a few
#very slow deliveries pull it up (the distribution is right-skewed).
#To halve the uncertainty we would need about four times as many deliveries.
#Limits: the sample is assumed to be random (a sample from only one city would not generalise).